In [ ]:
from transformers import AutoTokenizer,BitsAndBytesConfig, AutoModelForCausalLM
from peft import PeftModel
import torch
import pandas as pd
import re
from tqdm.notebook import tqdm
from prepare_data_for_dpo import extract_features_from_answer,generate_responses_batched
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
bnb_config_base_model=BitsAndBytesConfig(

    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
HF_TOKEN = os.getenv('HF_TOKEN')
model_name = "Qwen/Qwen3-4B"
tokenizer=AutoTokenizer.from_pretrained(model_name,token=HF_TOKEN)
tokenizer.pad_token=tokenizer.eos_token

In [ ]:
tokenizer=AutoTokenizer.from_pretrained('sft_model')
base_model_for_sft=AutoModelForCausalLM.from_pretrained(model_name,
                                                         device_map='auto',
                                                         quantization_config=bnb_config_base_model,
                                                         dtype=torch.float16,
                                                         token=HF_TOKEN
                                                        )
sft_model=PeftModel.from_pretrained(base_model_for_sft,'sft_model')
sft_model.eval()

In [ ]:
test_data=pd.read_csv('test_data_from_sft_data1')
test_data=extract_features_from_answer(test_data)
test_data_sample=test_data.sample(400)
test_data_sample_prompts=test_data_sample['user_message'].tolist()
test_data_sample_labels=test_data_sample['is_unsafe'].tolist()

In [ ]:
results=generate_responses_batched(sft_model,tokenizer,test_data_sample_prompts,test_data_sample_labels)

In [ ]:
results=pd.DataFrame(results)